# Exploration of Read v2 codes structure in prescriptions and clinical data

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [ ]:
hl.init(sc=sc, default_reference='GRCh38')

#### Envinroment setup

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Input database configuration and loading

In [ ]:
db_name = 'clinical_phenos'
full_tb_name = 'full_phenos_hail_0.2.116.ht'

In [ ]:
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database", project=dxpy.PROJECT_CONTEXT_ID)['id']
url = f"dnax://{db_uri}/{full_tb_name}"
full = hl.read_table(url)

### Checking dataset size

In [ ]:
full.count()

In [ ]:
%time system_df = full.filter(full.system == 'read_2').cache()
system_df.count()

### Adding neccessary helpers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

def perform_count_aggregation(grouped_by, input_data, aggregated = False):
    if aggregated:
        aggregated = grouped_by
    else:
        aggregated = grouped_by.aggregate(occurences=hl.agg.count())
    data_count = input_data.count()
    aggregated = aggregated.annotate(share = hl.format('%.3f%%', aggregated.occurences / hl.float(data_count) * 100))
    aggregated = aggregated.order_by(-aggregated.occurences).cache()
    return aggregated
    
def show_aggregated_examples(aggredated, source_data, column, sample_sz):
    source_data_cols = list(source_data.row.keys())
    source_data_cols.remove(column)
    source_data_cols.insert(0, column)
    aggregated_py = aggregated.collect()
    rand_seed = int(datetime.now().timestamp())
    source_data = source_data.annotate(rand = hl.rand_unif(0, 1, seed = rand_seed)).order_by('rand').cache()
    joined = source_data.head(0)
    for row in aggregated_py:
        joined = joined.union(source_data.filter(source_data[column] == row[column]).head(sample_sz))
    joined = joined.cache()
    joined = joined.key_by(column).join(aggredated.key_by(column), how = 'left')
    joined = joined.order_by(-joined.occurences).select(*source_data_cols).cache()
    joined.show(-1)

def aggregated_bar_plot(aggregated, column):
    aggregated_pd = aggregated.to_pandas()
    aggregated_pd['share_pct'] = aggregated_pd['share'].str.rstrip('%').astype(float)
    
    plt.figure(figsize=(8, 4))
    plt.bar(aggregated_pd[column], aggregated_pd['share_pct'], color = plt.cm.viridis(np.linspace(0, 1, 5)))
    plt.ylabel('Share %')

## Checking Read v2 codes length distribution

In [ ]:
%time system_df = system_df.annotate(code_len = hl.len(system_df.code)).cache()

In [ ]:
%time aggregated = perform_count_aggregation(system_df.group_by('code_len'), system_df)
aggregated.show(-1)

In [ ]:
# show_aggregated_examples(aggregated, system_df, 'code_len', 5)

In [ ]:
aggregated_bar_plot(aggregated.annotate(code_len = hl.str(aggregated.code_len)), 'code_len')
plt.title('Read v2 prescriptions code lenght')
plt.show()

## Checking Read v2 codes formats

**Notice**: Following findings about Read v2 codes structure are mainly data-driven (Biobank clinical and prescriptions data and Read lookups/dictionaries).

Read v2 codes format in Biobank data falls into a few categories:


| Class name           | Format as regular expression                     | Example   | Description                                                                 |
|----------------------|--------------------------------------------------|-----------|-----------------------------------------------------------------------------|
| `drug`                 | `^[a-z][A-Z0-9a-z]{3}\.$`                        | `bxd5.`   | Exact drug codes – 1 dot.                                                   |
| `drug_00_suffix`       | `^[a-z][A-Z0-9a-z][A-Z0-9a-z.]{2}\.00$`          | `bl8A.00` | Exact drug codes but with suspicious `.00` suffix.                         |
| `drug_substance`       | `^[a-z][A-Z0-9a-z]{2}\.\.$`                      | `a13..`   | Codes for substance (API) – 2 dots.                                        |
| `drug_category`        | `^[a-z][A-Z0-9a-z]\.\.\.$`                       | `a1...`   | Codes for broader drug category (i.e. "loop diuretics") – 3 dots.          |
| `clinical_letter`      | `^[A-Z][A-Z0-9a-z.]{4}$`                         | `N2410`   | Codes for clinical records – diseases and conditions.                      |
| `clinical_digit`       | `^[0-9][A-Z0-9a-z.]{4}$`                         | `65ED.`   | Other codes for clinical records – events, measurements, visits, etc.      |
| `blank`                | `^\.{5}$`                                        | `.....`   | Empty codes? Hard to tell.                                                 |


* There are no 5-bytes codes for drugs/prescriptions in the dataset.
* There are a few (n=3) prescriptions with 4-dot (1-byte) code but I omitted them in classification.
* There are a few records (n=6) from `gp_clinical` table with drug codes (prescription-like) but w/o descriptions.
* If a code ends with `.00` description filed is always present and record is for a drug.

In [ ]:
code_formats = {
    'drug': (r'^[a-z][A-Z0-9a-z]{3}\.$', 'bxd5.'),
    'drug_00_suffix': (r'^[a-z][A-Z0-9a-z][A-Z0-9a-z.]{2}\.00$', 'bl8A.00'),
    'drug_substance': (r'^[a-z][A-Z0-9a-z]{2}\.\.$', 'a13..'),
    'drug_category': (r'^[a-z][A-Z0-9a-z]\.\.\.$', 'a1...'),
    'clinical_letter': (r'^[A-Z][A-Z0-9a-z.]{4}$', 'N2410'),
    'clinical_digit': (r'^[0-9][A-Z0-9a-z.]{4}$', '65ED.'),
    'blank': (r'^\.{5}$', '.....'),
}

In [ ]:
%%time
hl_code_formats = hl.literal([(code_formats[k][0], k) for k in code_formats.keys()])
system_df = system_df.annotate(
    code_format = hl.or_else(hl.find(lambda cformat: system_df
.code.matches(cformat[0]), hl_code_formats), hl.literal(('.+', 'other')))[1]
).cache()

#### Examining unknown Read v2 code format records

In [ ]:
other = system_df.filter(system_df.code_format == 'other').cache()
other.count()

In [ ]:
other.show()

#### Read v2 code formats records share

In [ ]:
%time aggregated = perform_count_aggregation(system_df.group_by('code_format'), system_df)
aggregated.show(-1)

In [ ]:
aggregated_bar_plot(aggregated, 'code_format')
plt.title('Read v2 code formats in data')
plt.xticks(rotation=60)
plt.show()

#### Records examples of particular Read v2 code formats

In [ ]:
%time show_aggregated_examples(aggregated, system_df, 'code_format', 5)